# Load PyTorch Chunk Files (`.pt`)

This notebook loads and inspects mixed human/AI datasets saved by `main.py`.

Configuration is read from `.env`:
- `MINIPILE_OUTPUT_DIR` — where `.pt` chunks are stored
- `GENERATION_MODE` — `append`, `sandwitch`, or `sandwitch_v2`

Each `.pt` file contains:
- `texts` — full mixed text
- `labels` — word-level labels (`0` = human, `1` = AI)
- `models` — OpenRouter model used per sample
- `indices` — filtered MiniPile stream index (same as `main.py --from`)
- `min_words_filter`, `generation_mode` — optional metadata on newer chunks

| Mode | Structure | Label pattern |
|------|-----------|---------------|
| `append` | human prefix + AI continuation | `000...111...` |
| `sandwitch` | human start + AI bridge + human end | `000...111...000...` |
| `sandwitch_v2` | summarize removed middle, regenerate AI middle | `000...111...000...` |

In [66]:
import os
from collections import Counter
from pathlib import Path
import random

import pyarrow.parquet as pq
from dotenv import load_dotenv

from load_pt import (
    MixedTextChunkDataset,
    list_chunks,
    load_chunk,
    load_split,
    lookup_original_for_sample,
    make_dataloader,
    summarize_chunk,
)

PROJECT_DIR = Path.cwd()
load_dotenv(PROJECT_DIR / ".env", override=True)


def env_value(key, default=""):
    value = os.environ.get(key, default)
    return value.split("#", 1)[0].strip()


GENERATION_MODE = env_value("GENERATION_MODE", "sandwitch")
output_dir = env_value("MINIPILE_OUTPUT_DIR", "output")
output_path = Path(output_dir)

if output_path.name in ("append", "sandwitch", "sandwitch_v2"):
    OUTPUT_DIR = output_path
elif output_path.name == "output" or output_dir == "output":
    OUTPUT_DIR = output_path / GENERATION_MODE
else:
    OUTPUT_DIR = output_path

if GENERATION_MODE == "sandwitch_v2":
    from sandwitch_v2 import MIN_SANDWITCH_V2_WORDS as MINIPILE_MIN_WORDS
elif GENERATION_MODE == "sandwitch":
    from sandwitch_sequence import MIN_SANDWITCH_WORDS as MINIPILE_MIN_WORDS
else:
    MINIPILE_MIN_WORDS = 3

DATA_DIR = Path(env_value("MINIPILE_DATA_DIR", "dataset/minipile/data"))
SPLIT = env_value("MINIPILE_SPLITS", "train").split(",")[0]
FROM_IDX = int(env_value("MINIPILE_FROM", "0"))
TO_IDX = int(env_value("MINIPILE_TO")) if env_value("MINIPILE_TO") else None

print(f"Output dir:       {OUTPUT_DIR}")
print(f"Generation mode:  {GENERATION_MODE}")
print(f"Data dir:         {DATA_DIR}")
print(f"Split:            {SPLIT}")
print(f"Index range:      [{FROM_IDX}, {TO_IDX})")


def detect_mode(labels) -> str:
    labels = [int(x) for x in labels]
    if 1 not in labels:
        return "unknown"
    last_one = len(labels) - 1 - labels[::-1].index(1)
    trailing_human = any(label == 0 for label in labels[last_one + 1 :])
    if trailing_human:
        return "sandwitch_v2" if GENERATION_MODE == "sandwitch_v2" else "sandwitch"
    return "append"


def split_by_labels(text: str, labels, mode=None):
    words = text.split()
    labels = [int(x) for x in labels]
    if len(words) != len(labels):
        raise ValueError(
            f"Word count ({len(words)}) != label count ({len(labels)}); text/labels misaligned."
        )

    mode = mode or detect_mode(labels)
    segments = {"human_start": [], "ai": [], "human_end": []}

    for word, label in zip(words, labels):
        if label == 1:
            segments["ai"].append(word)
        elif segments["ai"]:
            segments["human_end"].append(word)
        else:
            segments["human_start"].append(word)

    if mode == "append":
        segments["human_end"] = []
    return mode, segments


def source_human_segments(original: str, mode: str, segments):
    """Human segments from the original text (same rules as generation)."""
    original_words = original.split()
    if mode == "append":
        n = len(segments["human_start"])
        return {
            "human_start": original_words[:n],
            "human_end": [],
            "skipped_middle": original_words[n:],
        }
    if mode == "sandwitch_v2":
        from sandwitch_v2 import split_into_three_parts

        begin, middle, end = split_into_three_parts(original)
        return {
            "human_start": begin.split(),
            "human_end": end.split(),
            "skipped_middle": middle.split(),
        }
    n_start = len(segments["human_start"])
    n_end = len(segments["human_end"])
    return {
        "human_start": original_words[:n_start],
        "human_end": original_words[-n_end:],
        "skipped_middle": (
            original_words[n_start:-n_end]
            if n_end and n_start + n_end < len(original_words)
            else original_words[n_start:]
        ),
    }


def source_split_mode(detected: str) -> str:
    """Use .env / chunk metadata for how the original was split at generation time."""
    if GENERATION_MODE in ("append", "sandwitch", "sandwitch_v2"):
        return GENERATION_MODE
    return detected


def compare_with_original(item, data_dir: Path, split: str, chunk_data=None):
    detected = detect_mode(item["labels"])
    mode, segments = split_by_labels(item["text"], item["labels"], mode=detected)
    split_mode = source_split_mode(detected)

    original, resolved_index, min_words_used, lookup_mode = lookup_original_for_sample(
        item,
        data_dir,
        split,
        chunk_data=chunk_data,
        default_min_words=MINIPILE_MIN_WORDS,
        human_start_words=segments["human_start"][:20],
    )
    source = source_human_segments(original, split_mode, segments)

    print(f"Stored index: {item['index']}")
    if resolved_index != item["index"]:
        print(f"Resolved index: {resolved_index} ({lookup_mode})")
    else:
        print(f"Resolved index: {resolved_index} ({lookup_mode}, min_words={min_words_used})")
    print(f"OpenRouter model: {item['model']}")
    print(f"Expected mode (.env): {GENERATION_MODE}")
    print(f"Detected mode (labels): {detected}")
    print(f"Source split mode: {split_mode}")
    print(
        f"Label counts: human_start={len(segments['human_start'])}, "
        f"ai={len(segments['ai'])}, human_end={len(segments['human_end'])}"
    )

    start_match = segments["human_start"] == source["human_start"]
    end_match = segments["human_end"] == source["human_end"]
    print(f"human_start matches source split: {start_match}")
    print(f"human_end matches source split: {end_match}")

    print("\n--- Original (same index as main.py)")
    print(original)

    if detected == "append":
        print(f"\n--- Original prefix [{len(source['human_start'])} words] ---")
        print(" ".join(source["human_start"]))
        print("\n--- Recovered human prefix (from labels) ---")
        print(" ".join(segments["human_start"]))
        print("\n--- AI continuation ---")
        print(" ".join(segments["ai"]))
    else:
        print("\n--- Recovered human_start ---")
        print(" ".join(segments["human_start"]))
        print("\n--- Source human_start (from original split) ---")
        print(" ".join(source["human_start"]))
        print("\n--- AI bridge ---")
        print(" ".join(segments["ai"]))
        if source["skipped_middle"]:
            print(f"\n--- Original middle removed at generation ({len(source['skipped_middle'])} words) ---")
            print(
                " ".join(source["skipped_middle"][:40])
                + ("..." if len(source["skipped_middle"]) > 40 else "")
            )
        print("\n--- Recovered human_end ---")
        print(" ".join(segments["human_end"]))
        print("\n--- Source human_end (from original split) ---")
        print(" ".join(source["human_end"]))

    return mode, segments


def show_labeled_segments(text: str, labels):
    mode, segments = split_by_labels(text, labels)
    print(f"Mode (.env): {mode}")
    print(
        f"human_start ({len(segments['human_start'])}): "
        f"{' '.join(segments['human_start'][:15])}{'...' if len(segments['human_start']) > 15 else ''}"
    )
    print(
        f"ai          ({len(segments['ai'])}): "
        f"{' '.join(segments['ai'][:15])}{'...' if len(segments['ai']) > 15 else ''}"
    )
    if segments["human_end"]:
        print(
            f"human_end   ({len(segments['human_end'])}): "
            f"{' '.join(segments['human_end'][:15])}{'...' if len(segments['human_end']) > 15 else ''}"
        )

ImportError: cannot import name 'get_filtered_minipile_text' from 'load_pt' (/home/ine/projects/human-ai-mixed-data-generator/load_pt.py)

## 1. List available chunk files

In [ ]:
chunks = list_chunks(OUTPUT_DIR, SPLIT)
print(f"Found {len(chunks)} chunk(s) for split '{SPLIT}':")
for path in chunks:
    print(f"  {path}")

Found 5 chunk(s) for split 'train':
  output/sandwitch_v2/train/train_0_0.pt
  output/sandwitch_v2/train/train_0_99.pt
  output/sandwitch_v2/train/train_10000_10999.pt
  output/sandwitch_v2/train/train_1000_1999.pt
  output/sandwitch_v2/train/train_100_999.pt


## 2. Summarize chunk + pick a sample

In [67]:
PT_PATH = chunks[2] if chunks else OUTPUT_DIR / SPLIT / "train_0_999.pt"

info = summarize_chunk(PT_PATH)
print(f"Path:       {info['path']}")
print(f"Samples:    {info['samples']}")
print(f"Indices:    {info['index_range'][0]}–{info['index_range'][1]}")
print(f"Model set:  {', '.join(info['models'])}")

Path:       output/sandwitch_v2/train/train_10000_10999.pt
Samples:    1000
Indices:    10000–10999
Model set:  cohere/command-r7b-12-2024+cohere/command-r7b-12-2024, cohere/command-r7b-12-2024+deepseek/deepseek-v4-flash, cohere/command-r7b-12-2024+google/gemma-3-4b-it, cohere/command-r7b-12-2024+meta-llama/llama-3.1-8b-instruct, cohere/command-r7b-12-2024+mistralai/mistral-nemo, cohere/command-r7b-12-2024+mistralai/mistral-small-24b-instruct-2501, cohere/command-r7b-12-2024+sao10k/l3-lunaris-8b, deepseek/deepseek-v4-flash+cohere/command-r7b-12-2024, deepseek/deepseek-v4-flash+meta-llama/llama-3.1-8b-instruct, deepseek/deepseek-v4-flash+mistralai/mistral-nemo, deepseek/deepseek-v4-flash+mistralai/mistral-small-24b-instruct-2501, deepseek/deepseek-v4-flash+sao10k/l3-lunaris-8b, google/gemma-3-4b-it+cohere/command-r7b-12-2024, google/gemma-3-4b-it+meta-llama/llama-3.1-8b-instruct, google/gemma-3-4b-it+mistralai/mistral-nemo, google/gemma-3-4b-it+mistralai/mistral-small-24b-instruct-2501,

## 3. Load one chunk directly

In [68]:
data = load_chunk(PT_PATH)
print("Keys:", list(data.keys()))
print("Sample count:", len(data["texts"]))

Keys: ['texts', 'labels', 'models', 'indices']
Sample count: 1000


In [69]:
len(chunks)

5

## 4. Compare with original text (append or sandwitch)

In [70]:
TARGET_IN_CHUNK = random.randint(0, len(MixedTextChunkDataset(PT_PATH)) - 1)
item = MixedTextChunkDataset(PT_PATH)[TARGET_IN_CHUNK]
chunk_data = load_chunk(PT_PATH)
mode, segments = compare_with_original(item, DATA_DIR, SPLIT, chunk_data=chunk_data)

words = item["text"].split()
labels = [int(x) for x in item["labels"]]
print(f"\nwords == labels: {len(words) == len(labels)}")
print("\n--- Full mixed text ---")
print(item["text"])
print("\n--- Labels ---")
print(labels)

Index: 10142
OpenRouter model: cohere/command-r7b-12-2024+cohere/command-r7b-12-2024
Expected mode (.env): sandwitch_v2
Detected mode (labels): sandwitch
Label counts: human_start=532, ai=107, human_end=0

--- Original
There is fixin' to be a local brewery around here that I am planning to go visit, but it will be the first one and they will be draft only to start. Otherwise, the best beer you are getting at the package store is Negro Modelo.

I also brought a keg of my Belgian White and a homebrew I made spiced with ginger, honey and lemon peel in a corny keg.

--- AI bridge ---
Color Pattern Lions are a tawny or light brown color. The backs of their ears are a darker brown or black, and they are the only cats with dark tufts of hair on their tails. Male lion manes can range in color from blonde to dark brown and can be a variety of thicknesses. Food Lions are obligate carnivores, meaning they must eat meat to survive. Lions eat a variety of wild animals in Africa, including zebras, a

## 5. Scan mode distribution in a chunk

In [12]:
ds = MixedTextChunkDataset(PT_PATH)
mode_counts = Counter(detect_mode(ds[i]["labels"]) for i in range(len(ds)))

print(f"Chunk: {PT_PATH}")
print(f"Total samples: {len(ds)}")
for mode, count in sorted(mode_counts.items()):
    print(f"  {mode}: {count} ({100 * count / len(ds):.1f}%)")

seen = set()
for i in range(len(ds)):
    mode = detect_mode(ds[i]["labels"])
    if mode in seen or mode == "unknown":
        continue
    seen.add(mode)
    print(f"\nExample {mode} sample (index {ds[i]['index']}):")
    show_labeled_segments(ds[i]["text"], ds[i]["labels"])

Chunk: output/append/train/train_9000_9999.pt
Total samples: 1000
  append: 1000 (100.0%)

Example append sample (index 9000):
Mode (.env): append
human_start (96): The U.S. envoy for efforts to end the Ukrainian conflict, Kurt Volker, is set to...
ai          (60): Since the conflict began in 2014, thousands of people have been killed and over a...


## 6. Compare append vs sandwitch across chunks

In [13]:
for path in chunks[:3] + chunks[-1:]:
    chunk_ds = MixedTextChunkDataset(path)
    counts = Counter(detect_mode(chunk_ds[i]["labels"]) for i in range(len(chunk_ds)))
    print(f"\n{path.name}: append={counts.get('append', 0)}, sandwitch={counts.get('sandwitch', 0)}")


train_0_999.pt: append=1000, sandwitch=0

train_1000_1999.pt: append=1000, sandwitch=0

train_2000_2999.pt: append=1000, sandwitch=0

train_9000_9999.pt: append=1000, sandwitch=0


## 7. Load a split with index range filter

In [14]:
subset = load_split(OUTPUT_DIR, SPLIT, from_idx=FROM_IDX, to_idx=TO_IDX)
print(f"Subset size: {len(subset)}")
print(f"First index: {subset[0]['index']}")
print(f"Last index:  {subset[-1]['index']}")

Subset size: 1000
First index: 0
Last index:  999


## 8. DataLoader with padded labels

In [15]:
loader = make_dataloader(
    OUTPUT_DIR,
    SPLIT,
    batch_size=4,
    shuffle=False,
    from_idx=FROM_IDX,
    to_idx=TO_IDX,
)

batch = next(iter(loader))
print("Batch keys:", batch.keys())
print("Texts:", len(batch["text"]))
print("Labels shape:", batch["labels"].shape)
print("Indices:", batch["index"].tolist())
print("Models:", batch["model"])

Batch keys: dict_keys(['text', 'labels', 'model', 'index'])
Texts: 4
Labels shape: torch.Size([4, 144])
Indices: [0, 1, 2, 3]
Models: ['openrouter/owl-alpha', 'openrouter/owl-alpha', 'openrouter/owl-alpha', 'openai/gpt-oss-20b:free']


## 9. Inspect labeled segments

In [16]:
sample_paths = [chunks[0], chunks[-1]] if len(chunks) >= 2 else chunks
for path in sample_paths:
    if path is None or not path.exists():
        continue
    chunk_ds = MixedTextChunkDataset(path)
    sample = chunk_ds[0]
    print(f"\n=== {path.name} | index {sample['index']} | mode {GENERATION_MODE} ===")
    show_labeled_segments(sample["text"], sample["labels"])


=== train_0_999.pt | index 0 | mode append ===
Mode (.env): append
human_start (91): HTC's Vive Pro headset is available to pre-order for $799 We've seen plenty of Beats-focused...
ai          (48): pair delivers surprisingly rich audio quality that belies its playful appearance. The sound profile leans...

=== train_9000_9999.pt | index 9000 | mode append ===
Mode (.env): append
human_start (96): The U.S. envoy for efforts to end the Ukrainian conflict, Kurt Volker, is set to...
ai          (60): Since the conflict began in 2014, thousands of people have been killed and over a...
